# Load & Compress Data Pipeline

This notebook loads site revenue data and joins it with site characteristics to produce a combined dataset.

**Pipeline stages:**
1. Load 7 key columns from `site_scores_revenue_and_diagnostics.csv` (date, id_gbase, month, year, avg_latency, revenue, statuis)
2. Aggregate monthly records → one row per site (revenue totals/averages, latency, status)
3. Load `sites_base_data_set.csv` (site characteristics, restrictions, capabilities)
4. Join aggregated revenue data with site characteristics on `id_gbase`
5. Export combined dataset as `site_revenue_diagnostics_and_characteristics.csv`

**Input:** `site_scores_revenue_and_diagnostics.csv` (~927 MB) + `sites_base_data_set.csv` (~17 MB)
**Output:** `site_revenue_diagnostics_and_characteristics.csv`

In [ ]:
import polars as pl
import numpy as np
from pathlib import Path
from datetime import datetime, date
from typing import Optional, List, Tuple

# Project paths
PROJECT_ROOT = Path('.').resolve()
INPUT_DIR = PROJECT_ROOT / 'data' / 'input'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Input directory:  {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'\nInput files:')
for f in sorted(INPUT_DIR.iterdir()):
    if f.is_file() and f.suffix == '.csv':
        size_mb = f.stat().st_size / 1024 / 1024
        print(f'  {f.name:50s} {size_mb:8.1f} MB')

---
## Stage 1: Load Input Files

### 1a. Main Site Scores Data (7 columns only)

The application uses **Polars** (not pandas) for loading because the main CSV is 927 MB / 1.4M rows.
Polars is 10-20x faster than pandas for this workload (ADR-003).

We load only 7 key columns: `date`, `id_gbase`, `month`, `year`, `avg_latency`, `revenue`, `statuis`.
Site characteristics (demographics, restrictions, capabilities) come from `sites_base_data_set.csv` instead.

In [ ]:
# Load the main Site Scores CSV — only key columns needed for aggregation
# Site characteristics (demographics, restrictions, capabilities) come from sites_base_data_set.csv
# gtvid is included for downstream distance joins (GTVID key)
LOAD_COLUMNS = ['date', 'id_gbase', 'gtvid', 'month', 'year', 'avg_latency', 'revenue', 'statuis']

print('Loading Site Scores data (8 columns)...')
site_scores = pl.read_csv(
    INPUT_DIR / 'site_scores_revenue_and_diagnostics.csv',
    columns=LOAD_COLUMNS,
    infer_schema_length=10000,
    null_values=['', 'NA', 'null', 'Unknown']
)
print(f'Loaded {len(site_scores):,} monthly records for {site_scores["id_gbase"].n_unique():,} unique sites')
print(f'Columns ({len(site_scores.columns)}): {site_scores.columns}')
print(f'Date range: {site_scores["date"].min()} to {site_scores["date"].max()}')
site_scores.head(3)

### 1b. Auxiliary Geospatial Distance Files

Six auxiliary CSVs provide distance-to-landmark features (in miles).

In [ ]:
# Load auxiliary geospatial data — same as data_transform.load_auxiliary_data()
print('Loading auxiliary geospatial data...\n')

# 1. Nearest site distances
nearest = pl.read_csv(INPUT_DIR / 'nearest_site_distances.csv')
print(f'Nearest sites:         {len(nearest):,} records | cols: {nearest.columns}')

# 2. Interstate distances — some sites have multiple, take minimum distance
interstate_raw = pl.read_csv(INPUT_DIR / 'site_interstate_distances.csv')
interstate = interstate_raw.group_by('GTVID').agg([
    pl.col('distance_to_interstate_mi').min().alias('min_distance_to_interstate_mi'),
    pl.col('nearest_interstate').first().alias('nearest_interstate')
])
print(f'Interstate distances:   {len(interstate):,} unique sites (from {len(interstate_raw):,} raw records)')

# 3. Kroger distances (pre-aggregated — one row per site)
kroger = pl.read_csv(INPUT_DIR / 'site_kroger_distances.csv')
print(f'Kroger distances:      {len(kroger):,} records')

# 4. McDonald's distances (pre-aggregated — one row per site)
mcdonalds = pl.read_csv(INPUT_DIR / 'site_mcdonalds_distances.csv')
print(f"McDonald's distances:  {len(mcdonalds):,} records")

# 5. Walmart distances
walmart = pl.read_csv(INPUT_DIR / 'site_walmart_distances.csv')
print(f'Walmart distances:     {len(walmart):,} records')

# 6. Target distances
target = pl.read_csv(INPUT_DIR / 'site_target_distances.csv')
print(f'Target distances:      {len(target):,} records')

---
## Stage 2: Aggregate Monthly Records → One Row Per Site

With only 7 columns loaded, the aggregation is streamlined:
- **active_months**: Count of monthly records per site
- **first_month / last_month**: Date range of activity
- **status**: Most recent site status (from `statuis` column — typo in source)
- **total_revenue / avg_monthly_revenue**: Revenue metrics
- **avg_latency**: Mean latency across all active months

In [ ]:
# Build aggregation expressions — simplified for 8-column input
# Site metadata (demographics, restrictions, capabilities) comes from sites_base_data_set.csv
agg_exprs = [
    # Count of active months
    pl.len().alias('active_months'),
    # Date range
    pl.col('date').min().alias('first_month'),
    pl.col('date').max().alias('last_month'),
    # Site identifiers (last value)
    pl.col('gtvid').last().alias('gtvid'),
    # Most recent status (note: typo in source data — 'statuis' not 'status')
    pl.col('statuis').last().alias('status'),
    # Revenue total
    pl.col('revenue').sum().alias('total_revenue'),
    # Average latency across all months
    pl.col('avg_latency').mean().alias('avg_latency'),
]

print(f'Aggregating {len(site_scores):,} monthly records to one row per site...')
print(f'Using {len(agg_exprs)} aggregation expressions')

In [ ]:
# Sort by site and date, then aggregate
df_sorted = site_scores.sort(['id_gbase', 'date'])
site_agg = df_sorted.group_by('id_gbase').agg(agg_exprs)

# Calculate average monthly revenue
site_agg = site_agg.with_columns(
    (pl.col('total_revenue') / pl.col('active_months')).alias('avg_monthly_revenue')
)

print(f'Aggregated to {len(site_agg):,} sites')
print(f'Columns: {site_agg.columns}')

# Show status distribution
status_dist = site_agg.group_by('status').agg(pl.len().alias('count')).sort('count', descending=True)
print(f'\nStatus distribution:')
for row in status_dist.iter_rows(named=True):
    pct = row['count'] / len(site_agg) * 100
    print(f'  {row["status"]:20s} {row["count"]:6,} ({pct:.1f}%)')

---
## Stage 3: Join with Site Characteristics

The `sites_base_data_set.csv` contains per-site metadata previously embedded in the main CSV:
- **Restriction flags** (`R - *`): 29 advertising category restrictions per site
- **Capability flags** (`C - *`): 10 site capabilities (beer, lottery, diesel, NFC, etc.)
- **Demographics**: Average Household Income, Median Age
- **Average Daily Impressions**: Site-level impression metric

**Join key:** `id_gbase` (aggregated) ↔ `ID - Gbase` (base data set)

In [ ]:
# Load site characteristics
print('Loading sites_base_data_set.csv...')
sites_base = pl.read_csv(
    INPUT_DIR / 'sites_base_data_set.csv',
    infer_schema_length=10000,
    null_values=['', 'NA', 'null', 'Unknown']
)
print(f'Loaded {len(sites_base):,} sites with {len(sites_base.columns)} columns')
print(f'Unique sites (ID - Gbase): {sites_base["ID - Gbase"].n_unique():,}')

# Show column categories
r_cols = [c for c in sites_base.columns if c.startswith('R - ')]
c_cols = [c for c in sites_base.columns if c.startswith('C - ')]
other_cols = [c for c in sites_base.columns if not c.startswith('R - ') and not c.startswith('C - ')]
print(f'\nColumn categories:')
print(f'  Restriction flags (R - *): {len(r_cols)}')
print(f'  Capability flags (C - *):  {len(c_cols)}')
print(f'  Other fields:              {len(other_cols)} — {other_cols}')

# Join aggregated revenue data with site characteristics
combined = site_agg.join(
    sites_base,
    left_on='id_gbase',
    right_on='ID - Gbase',
    how='left'
)

print(f'\nCombined dataset: {len(combined):,} sites × {len(combined.columns)} columns')

# Check join coverage (R - Lottery has 0% missing in base data)
matched = combined.filter(pl.col('R - Lottery').is_not_null()).shape[0]
unmatched = len(combined) - matched
print(f'Join coverage: {matched:,} / {len(combined):,} ({matched/len(combined)*100:.1f}%)')
if unmatched > 0:
    print(f'Unmatched sites: {unmatched:,} (aggregated sites not found in base data set)')

combined.head(3)

In [ ]:
# Export combined dataset
output_path = OUTPUT_DIR / 'site_revenue_diagnostics_and_characteristics.csv'
combined.write_csv(output_path)
size_mb = output_path.stat().st_size / 1024 / 1024

print(f'Saved: {output_path.name} ({size_mb:.1f} MB)')
print(f'Shape: {len(combined):,} rows × {len(combined.columns)} columns')

# Show all columns with types and null counts
print(f'\nAll columns:')
agg_cols = ['id_gbase', 'active_months', 'first_month', 'last_month', 'status',
            'total_revenue', 'avg_latency', 'avg_monthly_revenue']
for i, col in enumerate(combined.columns):
    source = 'agg' if col in agg_cols else 'base'
    dtype = combined[col].dtype
    nulls = combined[col].null_count()
    pct = nulls / len(combined) * 100
    print(f'  {i+1:3d}. [{source:4s}] {col:45s} {str(dtype):10s} nulls: {nulls:,} ({pct:.1f}%)')

---
## Stage 4: Join Geospatial Distance Features

Join the 6 auxiliary distance datasets to the combined data via `gtvid` ↔ `GTVID`.
These distance features were loaded in Stage 1b.

In [ ]:
# Join nearest site distances
combined = combined.join(
    nearest.select([
        'GTVID',
        'nearest_site',
        pl.col('nearest_site_distance_mi').alias('min_distance_to_nearest_site_mi')
    ]),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join interstate distances
combined = combined.join(
    interstate,
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Kroger distances
combined = combined.join(
    kroger.select(['GTVID', 'min_distance_to_kroger_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join McDonald's distances
combined = combined.join(
    mcdonalds.select(['GTVID', 'min_distance_to_mcdonalds_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Walmart distances
combined = combined.join(
    walmart.select(['GTVID', 'min_distance_to_walmart_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Join Target distances
combined = combined.join(
    target.select(['GTVID', 'min_distance_to_target_mi']),
    left_on='gtvid',
    right_on='GTVID',
    how='left'
)

# Report join coverage
print('Geospatial feature join coverage:')
for col_name, label in [
    ('nearest_site', 'Nearest site'),
    ('min_distance_to_interstate_mi', 'Interstate'),
    ('min_distance_to_kroger_mi', 'Kroger'),
    ('min_distance_to_mcdonalds_mi', "McDonald's"),
    ('min_distance_to_walmart_mi', 'Walmart'),
    ('min_distance_to_target_mi', 'Target'),
]:
    if col_name in combined.columns:
        matched = combined.filter(pl.col(col_name).is_not_null()).shape[0]
        pct = matched / len(combined) * 100
        print(f'  {label:15s} {matched:6,} / {len(combined):,} ({pct:.1f}%)')

print(f'\nDataset with distances: {combined.shape[0]:,} sites × {combined.shape[1]} columns')

### Save Pre-cleaned Intermediate (optional checkpoint)

In [ ]:
# Save combined dataset with distances as checkpoint
precleaned_path = OUTPUT_DIR / 'site_aggregated_precleaned.parquet'
combined.write_parquet(precleaned_path)
size_mb = precleaned_path.stat().st_size / 1024 / 1024
print(f'Saved pre-cleaned checkpoint: {precleaned_path.name} ({size_mb:.1f} MB)')
print(f'Shape: {combined.shape[0]:,} sites × {combined.shape[1]} columns')

---
## Stage 4: Prepare Training Dataset

Transform the pre-cleaned data into ML-ready format:
1. Filter to **Active sites only**
2. Remove negative revenue records
3. Drop geographic identifiers (state, county, dma, zip)
4. **Bin high-cardinality categoricals** (retailer → top 30 + Other)
5. **Log-transform** numeric features
6. **One-hot encode** capability and restriction flags

In [ ]:
# Start with the combined dataset (aggregated revenue + sites_base + distances)
train_df = combined.clone()

# 4a. Filter to Active sites only
original_count = len(train_df)
train_df = train_df.filter(pl.col('status') == 'Active')
print(f'Filtered to Active sites: {len(train_df):,} (from {original_count:,})')

# 4b. Remove negative revenue records
before = len(train_df)
train_df = train_df.filter(pl.col('total_revenue') >= 0)
removed = before - len(train_df)
if removed > 0:
    print(f'Removed {removed:,} negative revenue records: {len(train_df):,} remaining')
else:
    print(f'No negative revenue records found')

# 4c. Drop identifier columns not needed for training
id_cols_to_drop = ['id_gbase', 'gtvid', 'first_month', 'last_month',
                   'nearest_site', 'nearest_interstate']
existing_id_cols = [c for c in id_cols_to_drop if c in train_df.columns]
train_df = train_df.drop(existing_id_cols)
print(f'Dropped identifier columns: {existing_id_cols}')

In [ ]:
# Note: retailer, brand_c_store, brand_fuel are not in the combined dataset
# (they were previously pulled from the 94-column site_scores CSV).
# The sites_base data provides restriction/capability flags instead.
print('No high-cardinality categorical columns to bin in the combined dataset')
print(f'Current shape: {train_df.shape[0]:,} sites × {train_df.shape[1]} columns')

In [ ]:
# 4e. Add log transformations for skewed numeric features
numeric_cols_to_log = [
    'total_revenue',
    'min_distance_to_nearest_site_mi', 'min_distance_to_interstate_mi',
    'min_distance_to_kroger_mi', 'min_distance_to_mcdonalds_mi',
    'min_distance_to_walmart_mi', 'min_distance_to_target_mi',
    'Average Household Income',
]

log_count = 0
for col in numeric_cols_to_log:
    if col in train_df.columns:
        safe_name = col.lower().replace(' ', '_').replace('-', '_')
        col_dtype = train_df[col].dtype
        if col_dtype in [pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64]:
            train_df = train_df.with_columns(
                (pl.col(col).cast(pl.Float64) + 1).log().alias(f'log_{safe_name}')
            )
        else:
            train_df = train_df.with_columns(
                pl.when(pl.col(col).cast(pl.Float64) >= 0)
                .then((pl.col(col).cast(pl.Float64) + 1).log())
                .otherwise(-(-pl.col(col).cast(pl.Float64) + 1).log())
                .alias(f'log_{safe_name}')
            )
        log_count += 1

print(f'Added {log_count} log-transformed columns')

In [ ]:
# 4f. Encode restriction (R - *) and capability (C - *) flags from sites_base
r_cols = [c for c in train_df.columns if c.startswith('R - ')]
c_cols = [c for c in train_df.columns if c.startswith('C - ')]

encoded_count = 0
for col in r_cols + c_cols:
    if col not in train_df.columns:
        continue
    # Create safe encoded column name
    prefix = 'r_' if col.startswith('R - ') else 'c_'
    safe_name = prefix + col.split(' - ', 1)[1].strip().lower()
    safe_name = safe_name.replace(' ', '_').replace('-', '_').replace('&', 'and')
    safe_name = safe_name.replace('/', '_').replace('(', '').replace(')', '')

    col_dtype = train_df[col].dtype

    if col_dtype == pl.Boolean:
        train_df = train_df.with_columns(
            pl.col(col).cast(pl.Int8).alias(f'{safe_name}_encoded')
        )
        encoded_count += 1
    elif col_dtype == pl.Utf8:
        unique_vals = set(train_df[col].drop_nulls().unique().to_list())
        if unique_vals <= {'Yes', 'No', 'Unknown'}:
            train_df = train_df.with_columns(
                pl.when(pl.col(col) == 'Yes').then(1)
                .when(pl.col(col) == 'No').then(0)
                .otherwise(None)
                .cast(pl.Int8)
                .alias(f'{safe_name}_encoded')
            )
            encoded_count += 1
        else:
            print(f'  Skipping {col}: unexpected values {unique_vals}')

# Drop original flag columns (keep encoded versions)
cols_to_drop = [c for c in r_cols + c_cols if c in train_df.columns]
train_df = train_df.drop(cols_to_drop)

print(f'Created {encoded_count} encoded flag columns, dropped {len(cols_to_drop)} original columns')
print(f'\nTraining dataset: {train_df.shape[0]:,} sites × {train_df.shape[1]} columns')

---
## Stage 5: Inspect & Export to Parquet

In [ ]:
# Inspect the final dataset
print('=== Final Training Dataset ===')
print(f'Shape: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'\nColumn breakdown:')
log_cols = [c for c in train_df.columns if c.startswith('log_')]
enc_cols = [c for c in train_df.columns if '_encoded' in c]
dist_cols = [c for c in train_df.columns if 'distance' in c.lower()]

print(f'  Log-transformed:   {len(log_cols)}')
print(f'  Encoded flags:     {len(enc_cols)}')
print(f'  Distance features: {len(dist_cols)}')

# Null check
null_counts = train_df.null_count()
cols_with_nulls = [(col, null_counts[col][0]) for col in null_counts.columns if null_counts[col][0] > 0]
if cols_with_nulls:
    print(f'\nColumns with nulls ({len(cols_with_nulls)}):')
    for col_name, count in sorted(cols_with_nulls, key=lambda x: -x[1])[:10]:
        print(f'  {col_name}: {count:,} nulls')
else:
    print('\nNo null values found!')

train_df.head(3)

In [ ]:
# Export to Parquet
output_path = OUTPUT_DIR / 'site_training_data.parquet'
train_df.write_parquet(output_path)

# Also save CSV for inspection
csv_path = OUTPUT_DIR / 'site_training_data.csv'
train_df.write_csv(csv_path)

# Report
parquet_mb = output_path.stat().st_size / 1024 / 1024
csv_mb = csv_path.stat().st_size / 1024 / 1024

print(f'=== Export Results ===')
print(f'Output CSV:         {csv_mb:8.1f} MB')
print(f'Output Parquet:     {parquet_mb:8.1f} MB')
print(f'CSV → Parquet:      {csv_mb / parquet_mb:.1f}x compression')
print(f'\nFiles saved:')
print(f'  {output_path}')
print(f'  {csv_path}')

---
## Verification: Compare with Application Output

Sanity check that our notebook output matches what `data_transform.py` produces.

In [ ]:
# Quick verification: load the parquet we just wrote and inspect
verify_df = pl.read_parquet(output_path)

print(f'Verification read: {verify_df.shape[0]:,} sites × {verify_df.shape[1]} columns')
print(f'\nKey stats:')
print(f'  avg_monthly_revenue: median=${verify_df["avg_monthly_revenue"].median():,.0f}, '
      f'mean=${verify_df["avg_monthly_revenue"].mean():,.0f}')
print(f'  active_months: median={verify_df["active_months"].median():.0f}, '
      f'mean={verify_df["active_months"].mean():.1f}')
print(f'\nAll columns:')
for i, col in enumerate(sorted(verify_df.columns)):
    print(f'  {i+1:3d}. {col} ({verify_df[col].dtype})')